# 🔨 SkillForge v3 — Research Papers & Repos → AI Skill Files

**Agentic CLI tool. Three input types. Two LLM providers. Zero-cost mode.**

```
arXiv Paper  →  SkillForge  →  SKILL.md (400+ lines, every equation, every table)
GitHub Repo  →  SkillForge  →  SKILL.md (300-700 lines, annotated code, configs, math)
Local PDF    →  SkillForge  →  SKILL.md (full extraction with quality gate)
```

**What's new in v3:**
- 🤖 **Agentic quality loop** — validate → fix gaps → re-validate (up to 3 rounds)
- 🛡️ Gemini safety filter handling (no more crashes)
- ♻️ Rate limit retry with exponential backoff
- 📊 GitHub prompt extracts math from code comments + demands benchmark results

**GitHub**: [github.com/AnubhavBharadwaaj/skillforge-ai](https://github.com/AnubhavBharadwaaj/skillforge-ai)

---

## Step 1: Install SkillForge

Clones the repo from GitHub and installs dependencies. No Drive mount needed.

In [ ]:
# Clone SkillForge from GitHub
!git clone https://github.com/AnubhavBharadwaaj/skillforge-ai.git /content/skillforge-ai 2>/dev/null || \
    (cd /content/skillforge-ai && git pull)

# Install dependencies
!pip install anthropic PyMuPDF google-generativeai -q

# Verify
!echo "" && wc -l /content/skillforge-ai/skillforge.py && echo "✅ SkillForge v3 ready!"

## Step 2: Set Your API Key

**Option A (recommended):** Use Gemini — **completely free**. Get a key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)

**Option B:** Use Anthropic — higher quality, ~$0.30/paper. Needs credits at [console.anthropic.com](https://console.anthropic.com)

**Option C:** No LLM — extracts raw text only, zero API calls

In [ ]:
import os

# ════════════════════════════════════════
#  PICK YOUR PROVIDER — uncomment ONE block
# ════════════════════════════════════════

# ── Option A: Gemini (FREE) ──
PROVIDER = "gemini"
MODEL = "gemini-2.5-flash"  # or "gemini-2.5-pro" for higher quality
os.environ["GEMINI_API_KEY"] = ""  # ← Paste your Gemini key here

# ── Option B: Anthropic (paid, best quality) ──
# PROVIDER = "anthropic"
# MODEL = "claude-sonnet-4-20250514"
# os.environ["ANTHROPIC_API_KEY"] = ""  # ← Paste your Anthropic key here

# ── Option C: No LLM (free, raw text extraction only) ──
# PROVIDER = "none"
# MODEL = ""

# ────────────────────────────────────────

print(f"✅ Provider: {PROVIDER}")
if PROVIDER == "gemini" and not os.environ.get("GEMINI_API_KEY"):
    print("⚠️  No GEMINI_API_KEY set! Get one free at: https://aistudio.google.com/apikey")
elif PROVIDER == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY"):
    print("⚠️  No ANTHROPIC_API_KEY set! Get one at: https://console.anthropic.com")

---
## Step 3: Run SkillForge

Pick **any cell below** depending on your input type. Each one is independent.

### 3A: arXiv Paper → Skill File

In [ ]:
ARXIV_ID = "2103.13630"  # ← Change to any arXiv paper ID or URL
DOMAIN = ""              # ← Optional: "parametergolf", "imageclef", "kaggle"

# Build command
if PROVIDER == "none":
    cmd = f"python /content/skillforge-ai/skillforge.py --arxiv {ARXIV_ID} --no-llm --output /content/skills"
else:
    cmd = f"python /content/skillforge-ai/skillforge.py --arxiv {ARXIV_ID} --provider {PROVIDER} --model {MODEL} --output /content/skills"
    if DOMAIN:
        cmd += f" --domain {DOMAIN}"

print(f"Running: {cmd}\n")
!{cmd}

### 3B: GitHub Repo → Skill File

In [ ]:
GITHUB_URL = "https://github.com/wolfecameron/nanoMoE"  # ← Change to any repo
DOMAIN = ""  # ← Optional

if PROVIDER == "none":
    cmd = f"python /content/skillforge-ai/skillforge.py --github '{GITHUB_URL}' --no-llm --output /content/skills"
else:
    cmd = f"python /content/skillforge-ai/skillforge.py --github '{GITHUB_URL}' --provider {PROVIDER} --model {MODEL} --output /content/skills"
    if DOMAIN:
        cmd += f" --domain {DOMAIN}"

print(f"Running: {cmd}\n")
!{cmd}

### 3C: Local PDF → Skill File

Upload a PDF from your computer, or use one from Google Drive.

In [ ]:
# Option 1: Upload from your computer
from google.colab import files
uploaded = files.upload()
PDF_NAME = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {PDF_NAME}")

In [ ]:
# Option 2: Use a PDF from Google Drive (uncomment and edit path)
# from google.colab import drive
# drive.mount('/content/drive')
# PDF_NAME = "/content/drive/MyDrive/papers/my_paper.pdf"  # ← Edit this path

In [ ]:
DOMAIN = ""  # ← Optional

if PROVIDER == "none":
    cmd = f"python /content/skillforge-ai/skillforge.py --pdf '{PDF_NAME}' --no-llm --output /content/skills"
else:
    cmd = f"python /content/skillforge-ai/skillforge.py --pdf '{PDF_NAME}' --provider {PROVIDER} --model {MODEL} --output /content/skills"
    if DOMAIN:
        cmd += f" --domain {DOMAIN}"

print(f"Running: {cmd}\n")
!{cmd}

### 3D: Batch Mode — Multiple Sources At Once

In [ ]:
# Edit this list — one arXiv ID, GitHub URL, or PDF path per line
sources = """
2103.13630
1510.00149
https://github.com/wolfecameron/nanoMoE
""".strip()

with open("/content/sources.txt", "w") as f:
    f.write(sources)

print("Sources:")
!cat /content/sources.txt

if PROVIDER == "none":
    cmd = f"python /content/skillforge-ai/skillforge.py batch --list /content/sources.txt --no-llm --output /content/skills --delay 1"
else:
    cmd = f"python /content/skillforge-ai/skillforge.py batch --list /content/sources.txt --provider {PROVIDER} --model {MODEL} --output /content/skills --delay 5"

print(f"\nRunning: {cmd}\n")
!{cmd}

---
## Step 4: View Results

In [ ]:
import glob, os

skill_files = sorted(glob.glob("/content/skills/*/SKILL.md"))
raw_files = sorted(glob.glob("/content/skills/raw_*.md"))
all_files = skill_files + raw_files

print(f"📁 Generated {len(all_files)} file(s):\n")
for sf in all_files:
    size = os.path.getsize(sf)
    with open(sf) as f:
        lines = f.read().count('\n')
    print(f"  📄 {sf}  ({lines} lines, {size:,} bytes)")

if all_files:
    print(f"\n{'='*60}")
    print(f"Preview of: {all_files[0]}")
    print(f"{'='*60}\n")
    with open(all_files[0]) as f:
        content = f.read()
    print(content[:3000])
    if len(content) > 3000:
        print(f"\n... [{len(content):,} total chars — run next cell to see full output]")

In [ ]:
# View a specific skill file in full (change the path as needed)
SKILL_PATH = "/content/skills/"  # ← Edit: e.g. "/content/skills/deep-compression/SKILL.md"

# Auto-pick the first one if you didn't edit the path
if SKILL_PATH == "/content/skills/":
    import glob
    files = sorted(glob.glob("/content/skills/*/SKILL.md"))
    if files:
        SKILL_PATH = files[0]
        print(f"Showing: {SKILL_PATH}\n")

!cat "{SKILL_PATH}"

---
## Step 5: Download Results

In [ ]:
# Option A: Download as ZIP
from google.colab import files
!cd /content && zip -r skill_library.zip skills/
files.download("/content/skill_library.zip")

In [ ]:
# Option B: Copy to Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/skillforge-output
!cp -r /content/skills/* /content/drive/MyDrive/skillforge-output/
print("✅ Copied to Drive at: MyDrive/skillforge-output/")

---

## 💡 Tips

- **Free tier exhausted?** Create a new Google Cloud project at [console.cloud.google.com](https://console.cloud.google.com) for fresh Gemini quota
- **`gemini-2.5-flash`** is fastest and free; **`gemini-2.5-pro`** gives better extraction quality
- **`--skip-validation`** saves 1-3 API calls if you're hitting rate limits
- **`--domain parametergolf`** adds competition-specific assessment to the skill file
- **`--no-llm`** extracts raw text with zero API calls — paste into Claude/ChatGPT manually

---

⭐ **Star the repo** if this saved you time: [github.com/AnubhavBharadwaaj/skillforge-ai](https://github.com/AnubhavBharadwaaj/skillforge-ai)